# 27 — The Bell Curve engine (v1.0)

Statistics, probability and statistical mechanics in one place (`ValaQuenta/modules/bell_curve`). Everything here is ESTABLISHED mathematics; the engine's claim is one verified implementation. Cross-checked against scipy 1.18 in `Testing/test_bell_curve.py` (45 tests).

In [1]:
import sys, random; sys.path.insert(0,'../../..')
from ValaQuenta.modules.bell_curve import maths as M
M.verify_known_values()

{'norm_cdf(1.96)': True,
 'norm_ppf(0.975)': True,
 'three_sigma_p': True,
 'six_sigma_dpmo': True,
 'chi2_crit_1df_95': True,
 't_crit_10df_975': True,
 'ising_matches_bruteforce': True,
 'softmax_is_canonical': True}

## 1. Streaming moments, merged exactly
A gradient that is 'diluted or saturated' is accumulated here.

In [2]:
R=random.Random(1); xs=[R.expovariate(1) for _ in range(1000)]
a=M.Moments().extend(xs[:400]); b=M.Moments().extend(xs[400:])
print(a.merge(b).summary()); print(M.Moments().extend(xs).summary())

{'n': 1000, 'mean': 1.0410713366243673, 'std': 1.0022216895079767, 'sem': 0.031693032592672844, 'skew': 1.6813306086389836, 'kurt_excess': 3.3239542937462945}
{'n': 1000, 'mean': 1.041071336624367, 'std': 1.0022216895079765, 'sem': 0.03169303259267284, 'skew': 1.6813306086389852, 'kurt_excess': 3.3239542937463016}


## 2. The bell curve, and n-sigma

In [3]:
for s in (1,2,3,5,6): print(s, 'sigma  two-sided p =', M.sigma_to_p(s), ' DPMO(6σ shift 1.5) =' , round(M.sigma_to_dpmo(s),4))

1 sigma  two-sided p = 0.31731050786291415  DPMO(6σ shift 1.5) = 691462.4613
2 sigma  two-sided p = 0.04550026389635844  DPMO(6σ shift 1.5) = 308537.5387
3 sigma  two-sided p = 0.0026997960632601913  DPMO(6σ shift 1.5) = 66807.2013
5 sigma  two-sided p = 5.733031437583892e-07  DPMO(6σ shift 1.5) = 232.6291
6 sigma  two-sided p = 1.9731752900754024e-09  DPMO(6σ shift 1.5) = 3.3977


## 3. A two-sample test, effect size, and an honest interval

In [4]:
X=[R.gauss(0,1) for _ in range(200)]; Y=[R.gauss(0.3,1) for _ in range(200)]
print(M.t_test_welch(X,Y)); print('hedges g', M.hedges_g(X,Y)); print('wilson 7/40', M.wilson_ci(7,40))

{'t': -3.279208184262404, 'df': 397.3765516563834, 'p': 0.001132729523223519, 'diff': -0.33179447963435854, 'ci': (-0.5307119911536874, -0.13287696811502972), 'd': -0.3279208184262404}
hedges g -0.32730248878747314
wilson 7/40 (0.0874541374603592, 0.31949990331787714)


## 4. Statistical mechanics: softmax is the canonical ensemble
Temperature dilutes (high T) or saturates (low T) a mixture.

In [5]:
x=[1,2,3]
for T in (0.1,0.5,1,5,100): print(T, [round(p,3) for p in M.softmax(x,T)])
r=M.canonical([0,1,2.5,4],0.8,1.0); print({k:round(v,4) for k,v in r.items() if k!='p'})

0.1 [0.0, 0.0, 1.0]
0.5 [0.016, 0.117, 0.867]
1 [0.09, 0.245, 0.665]
5 [0.269, 0.329, 0.402]
100 [0.33, 0.333, 0.337]
{'Z': 1.3372, 'lnZ': 0.2906, 'U': 0.3166, 'F': -0.2325, 'S': 0.6863, 'Cv': 0.6251}


## 5. Circular statistics (for wheels, phases, hues)

In [6]:
import math
print(M.rayleigh_test([0.1,0.2,0.15,6.2,0.05]*6)); print(M.rayleigh_test([2*math.pi*i/50 for i in range(50)]))

{'R': 0.9952908371982657, 'Z': 29.71811551832474, 'p': 8.164432424461233e-22, 'mean_direction': 0.0834555206341144}
{'R': 5.039964043329555e-17, 'Z': 1.2700618779027401e-31, 'p': 1.0, 'mean_direction': -2.4686772424678995}


## Open items
No multivariate tests, no time-series (ARMA), no MCMC and no GLM yet; add when a pipeline asks.